# Case study: trimming redundant features before a model

A churn table arrives with nine columns and a deadline. The modeling team
wants to know which columns carry their own information — and which are
derived, duplicated, or concentrated. The profiler measures which
columns are derivable from others, how their values are distributed, and
which columns predict the target. The modeling team can use these results
to review candidate features.

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(77)
n = 5_000

tenure = [rng.randrange(0, 60) for _ in range(n)]
tenure_months = [round(t / 12, 2) for t in tenure]          # rounded copy
x1 = [rng.uniform(-1, 1) for _ in range(n)]
x2 = [rng.uniform(-1, 1) for _ in range(n)]
score = [round(a * b + rng.uniform(-0.05, 0.05), 4) for a, b in zip(x1, x2)]  # interaction
channel = ["web" if rng.random() < 0.98 else "app" for _ in range(n)]          # concentrated
segment = [rng.choice(["retail", "b2b"]) for _ in range(n)]
plan = [["starter", "scale"][s == "b2b"] if rng.random() < 0.97
        else ["starter", "scale"][1 - (s == "b2b")]
        for s in segment]                                    # 3% violations
# new customers churn: tenure under a year, with 5% label noise
churn = [(rng.random() < 0.95) == (t < 12) for t in tenure]

df = pl.DataFrame({
    "tenure": tenure,
    "tenure_months": tenure_months,
    "x1": x1,
    "x2": x2,
    "score": score,
    "channel": channel,
    "segment": segment,
    "plan": plan,
    "churn": churn,
})

report = onceover.profile(df)
print(report.overview["headline"])

The headline says the table is mostly structure: functional dependencies,
predictable columns, and rules. It also says `key candidate: x1` — which is a false lead for feature selection. `x1` is a random float, unique in all 5,000 rows;
uniqueness plus no nulls is exactly what the key rule measures, so the
ranking promotes it. The profiler reports the measurement, not the
intent. Read the key candidate as "this column is unique", not "this
column is an identifier".

For feature trimming, three kinds of statements matter — dependencies
(what is derivable), concentration (one-hot traps), and predictive
statements (what moves with what). Let us pull each kind.

In [ ]:
print("— dependencies")
for f in report.findings:
    if f["category"] == "dependencies":
        print(f"  [{f['type']}] {f['claim']}")
print()
print("— concentration and roles")
for f in report.findings:
    if f["category"] in {"concentration", "column_roles"}:
        print(f"  {f['claim']}")
print()
print("— rules")
for f in report.findings:
    if f["category"] == "conditional_rules":
        print(f"  {f['claim']}")

## The trimming checklist

**Redundant measurements.** `tenure` and `tenure_months` are exact
functions of each other: 0 violations in both directions. Because
`tenure_months` is `tenure / 12` rounded to two decimals, every observed
integer month count maps to one year value and back, so the profiler's
partition test finds a bijection rather than fitting a slope. Any model
learns the rescaling for free; keeping both splits importance between
two copies of one signal. Decision: drop `tenure_months`, keep `tenure`.

**A unique float is not a key.** `x1` and `x2` both surface as
identifier-role columns, and `x1` wins the headline's key ranking. In a
feature table that is a warning: the column is unique because it is
continuous, not because it identifies an entity. Keep it as a feature;
do not join on it.

**Interactions are not redundancy.** `score` is predictable from the rest
of the table at holdout R² 0.98, and the model's gain is almost entirely
on `x1` and `x2` (shares 0.63 and 0.37; every other input is below
0.001). The piecewise model makes the same point from another angle:
`score` follows 8 local linear regimes with holdout R² 0.910, against
0.002 for one global linear model. Pairwise correlation would have
called this relationship absent. Dropping either input kills the
feature; keeping `score` and engineering the product explicitly is the
cleaner option.

**Concentrated columns are usually noise.** `channel` is 98.1% one
value — effective cardinality 1.10. One-hot encoding it spends a column
to carry a constant. Decision: keep it out of the model unless the 2%
pocket is the point (it is not, here — the rules that mention it are
pockets of `app`, not structure).

**Rules describe pockets, not the model.** `segment → plan` holds in
96–98% of matching rows depending on direction (b2b → scale 96.03%,
retail → starter 98.41%); the profiler also lists `segment, score → plan`
at 99.22% with 39 violations. The second statement is stricter but not
better: `score` is a near-unique float, so putting it in the antecedent
nearly pins each row to itself. These are data-quality notes for joins
and dashboards, not modeling features.

**The target is predictable.** `churn` is learned
from `tenure` (with `x2` and `score` adding nothing): holdout accuracy
0.95 against a 0.77 majority baseline, and a cross-validated decision
table on `tenure` alone reaches 0.896 against 0.769. That is the signal
worth modeling.

In [ ]:
target = [f for f in report.findings
          if f["category"] == "dependencies" and f.get("type") == "predictive"
          and f["subject"].get("target") == "churn"]
for f in target:
    e = f["evidence"][0]
    m = e["metrics"]
    value = m.get("metric", m.get("cv_accuracy"))
    print(f["claim"])
    print("   inputs:", f["subject"]["inputs"],
          "| score:", round(value, 3),
          "| baseline:", round(m["baseline"], 3))

## The control: shuffle the target

As a negative control, shuffle `churn` and profile the table again. The
predictive findings about `churn` should disappear. If they remain,
investigate possible leakage in the analysis.

In [ ]:
permuted = df.with_columns(pl.col("churn").shuffle(seed=77))
permuted_report = onceover.profile(permuted)
kept = [f for f in permuted_report.findings
        if f["category"] == "dependencies" and f.get("type") == "predictive"
        and f["subject"].get("target") == "churn"]
print("predictive churn findings after shuffling:", len(kept))

Zero — the claim depended on the actual relationship, and destroying it
destroyed the finding. That is what `estimated`-basis findings should do
under a negative control, and it is cheap enough to run on any table
before trusting the screen's output.

## Summary

| Statement | Evidence class | Trimming decision |
|---|---|---|
| `tenure` and `tenure_months` are exact functions of each other (0 violations) | measured · full | drop the rounded copy |
| `score` predictable from the rest (holdout R² 0.98; 8 regimes, 0.910 vs 0.002 linear) | estimated · holdout | keep `x1`, `x2`; consider an explicit product |
| `channel` 98.1% one value | measured · full | exclude from one-hot |
| `segment → plan` 96–98%; `segment, score → plan` 99.22% | measured · diagnostic partition | dedupe dashboards, not a model feature |
| `churn` from `tenure` (0.896 CV; holdout 0.95 vs 0.77) | estimated · holdout | the table has signal; proceed |

The groups view ties it together — connected findings over shared
columns, without claiming interchangeability:

In [ ]:
for g in report.groups:
    print(g["id"], "—", len(g["columns"]), "columns,", len(g["finding_ids"]), "findings")
    print("   columns:", ", ".join(g["columns"]))
    print("   ", g["description"])

The whole table is one connected component: the exact copy joins `tenure`
to `tenure_months`, the predictive statements join `score` to `x1` and
`x2`, and `churn` and `channel` touch enough of the rest to pull them in.
That is the right lesson from the groups view. Connectivity is a reading
aid — *which findings talk about the same columns* — not a redundancy
verdict, and one derived column can merge a whole table into a single
component. The trimming decisions remain the analyst's, made on the
statements themselves.